In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, regexp_extract

## altar o UTF para latin-1 
def get_available_years() -> list[str]:
    """
    Retorna todos os anos existentes na Bronze.
    Exemplo:
        ['2023', '2024', '2025']
    """

    return sorted(
        folder.name.rstrip("/")
        for folder in dbutils.fs.ls(BRONZE_PATH)
        if folder.isDir()
    )


def read_csv_from_bronze(file_name: str) -> DataFrame:
    """
    Lê um arquivo CSV presente em todos os anos da Bronze.
    Adiciona a coluna ano_referencia automaticamente.
    """

    paths = [
        f"{BRONZE_PATH}/{year}/{file_name}"
        for year in get_available_years()
    ]

    return (
        spark.read
        .options(**CSV_OPTIONS)
        .csv(paths)
        .withColumn(
            "ano_referencia",
            regexp_extract(
                col("_metadata.file_path"),
                r"bronze/(\d{4})/",
                1
            )
        )
    )